In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import fasttext
import fasttext.util
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Definir un Dataset personalizado
class TextDataset(Dataset):
    def __init__(self, texts, labels, ft_model):
        self.texts = texts
        self.labels = labels
        self.ft_model = ft_model
        self.num_classes = len(set(labels))

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        # Obtener la representación vectorial de cada palabra y promediar
        words = text.split()
        word_vectors = [self.ft_model.get_word_vector(w) for w in words if w in self.ft_model.words]
        if len(word_vectors) == 0:
            word_vectors = [np.zeros(300)]  # Si no hay palabras, usar vector de ceros
        text_vector = np.mean(word_vectors, axis=0)
        return torch.tensor(text_vector, dtype=torch.float32), torch.tensor(label, dtype=torch.long)

In [ ]:
# Cargar modelo de fastText preentrenado
import fasttext.util
ft_model = fasttext.load_model('/content/cc.en.300.bin') # Tenéis que descargarlo antes

# Cargar el conjunto de datos newsgroups
newsgroups = fetch_20newsgroups(subset='train', categories=['alt.atheism', 'sci.space', 'comp.graphics'])
texts = newsgroups.data
labels = newsgroups.target

# Codificar etiquetas
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(labels)

In [ ]:

# Definir el modelo LSTM
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = x.unsqueeze(1)  # Agregar dimensión de secuencia
        lstm_out, _ = self.lstm(x)
        out = self.fc(lstm_out[:, -1, :])
        return out


In [ ]:
# Dividir el conjunto de datos en entrenamiento y prueba
train_texts, test_texts, train_labels, test_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)

dataset = TextDataset(train_texts, train_labels, ft_model)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Definir modelo, pérdida y optimizador
input_dim = 300
hidden_dim = 128
output_dim = len(set(labels))
model = LSTMClassifier(input_dim, hidden_dim, output_dim)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [ ]:

# Entrenamiento, bajar batch size, puede durar mucho
num_epochs = 1
for epoch in range(num_epochs):
    for i, (text_vecs, label) in enumerate(dataloader):
        print(f"{i}/{len(dataloader)}")
        optimizer.zero_grad()
        outputs = model(text_vecs)
        loss = criterion(outputs, label)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}')

print("Entrenamiento completado")
